# urai

> The chat conventions every backend shares.

In [ ]:
#| hide
from urai import *

Urai means “dialog” in Tamil. It defines the chat behaviour shared by every backend. [aidialog](https://github.com/AnswerDotAI/aidialog) defines message parts. A backend package such as [rishi](https://github.com/vedicreader/rishi) talks to an inference engine. Urai manages the conversation between those boundaries.

A backend implements one model step. Urai supplies model resolution, portable options, history, tools, streaming, callbacks, and usage accounting.

## Install

```sh
pip install urai
```

Urai does not include an inference backend. Install one separately, then import its backend module to register a runtime.


## Start a chat

The examples below assume a backend has registered a `remote` runtime. Pass its model name to `mk_chat`, then call the returned chat with a message:

```python
from urai import mk_chat, resp_text

chat = mk_chat('gpt-5.1', runtime='remote', sp='Answer briefly.', temp=0.2)
reply = chat('What is 6 times 7?')
print(resp_text(reply))
```

`chat.hist` contains the normalized conversation. The same history shape works with every backend.

```python
[(msg['role'], resp_text(msg)) for msg in chat.hist]
```

## Resolve a model before loading it

`resolve` identifies a registered runtime and context window without building a backend:

```python
from urai import resolve

spec = resolve('claude-sonnet-4.6', runtime='remote')
print(spec.runtime, spec.ctx, spec.local)

chat = mk_chat(spec)
```

A resolved `ModelSpec` can be cached, compared, and passed between layers.

## Configure one turn

Constructor options set defaults for the conversation. Generation options passed to a call apply only to that turn:

```python
reply = chat('Explain the hard part.', effort='high', temp=0.9)
```

Urai translates portable names such as `ctx` and `temp` to backend-specific names. Unsupported options produce a warning. Backend-only options belong in `extra`.

## Add a tool

A Python callable becomes a tool from its signature and docstring:

```python
def add(a: int, b: int) -> int:
    "Add two integers."
    return a + b

chat = mk_chat('gpt-5.1', runtime='remote', tools=[add], approve=lambda call: True)
reply = chat('Use the tool to add 20 and 22.')
print(resp_text(reply))
```

Approval runs before each local tool call. Provider-run tools are recorded but never executed locally.

## Modules

| module | contract |
|---|---|
| `core` | usage, callbacks, and response rendering |
| `tags` | `<think>` and `<tool_call>` parsing |
| `msgs` | canonical messages and wire conversions |
| `caps` | model input, output, and context capabilities |
| `opts` | model resolution and portable options |
| `chat` | synchronous and asynchronous conversation state |
| `loop` | tool approval, execution, budgets, and context recovery |
| `eval` | classification, structured output, and grading |
| `sandbox` | Python-fence execution |
| `record` | deterministic recording and replay |
| `broker` | isolated conversations over one shared engine |

## Write a backend

Subclass `Chat`, declare option translations, and register the runtime. Add `ToolLoopMixin` when the backend returns tool calls as data.

```python
from urai import Chat, ToolLoopMixin, Runtime, register_runtime

class MyChat(ToolLoopMixin, Chat):
    _runtime = 'mine'
    _opt_map = {'ctx': 'n_ctx'}
    _opt_skip = ('effort',)

    def _model_step(self, **kw): ...
    def _stream_step(self, **kw): ...

register_runtime(Runtime('mine', 'mypkg.chat.MyChat', pats=('.mine',)))
```

`Chat('model.mine')` now dispatches to `MyChat`.

## Develop

The notebooks in `nbs/` are the source. Files in `urai/` are generated.

```sh
uv sync --all-extras --group dev
uv run nbdev-export
uv run nbdev-test
uv run nbdev-clean
```

The test suite uses scripted backends. It does not load a model or require network access.